In [1]:
import sys
sys.path.append("..")
from functools import partial
import pandas as pd
from src.data import load_cuad, TARGET_CLAUSES
from src.split import split_contracts
from src.chunking import clause_chunks, overlaps
from src.retrieval import make_dense_ranker
from src.bm25 import make_bm25_ranker
from src.hybrid import make_hybrid_ranker

contracts = load_cuad()
dev_idx, test_idx = split_contracts(contracts)

chunker = partial(clause_chunks, max_size=1500)
dense = make_dense_ranker(chunker, tag="clause-200-1500-200")
bm25 = make_bm25_ranker(chunker)
hybrid = make_hybrid_ranker([dense, bm25], weights=[1.0, 2.0])

In [2]:
def first_correct_rank(ranker, clause, idx):
    rows = []
    for i in idx:
        c = contracts[i]
        gold = c["gold"][clause]
        if not gold:
            continue
        ranked = ranker(c, clause)
        rank = next(r for r, ch in enumerate(ranked, 1) if overlaps(ch, gold))
        rows.append({"i": i, "title": c["title"], "rank": rank})
    return pd.DataFrame(rows)

nc = first_correct_rank(hybrid, "Non-Compete", dev_idx)
print("rank 1:", (nc["rank"] == 1).mean().round(2))
print("rank 2-3:", nc["rank"].between(2, 3).mean().round(2))
print("rank 4-10:", nc["rank"].between(4, 10).mean().round(2))
print("rank 11+:", (nc["rank"] > 10).mean().round(2))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

rank 1: 0.43
rank 2-3: 0.32
rank 4-10: 0.13
rank 11+: 0.12


In [3]:
def show(i, clause, ranker, n=700):
    c = contracts[i]
    s, e = c["gold"][clause][0]
    top = ranker(c, clause)[0]
    print(c["title"])
    print("GOLD :", c["text"][s:e][:n].replace("\n", " "))
    print("TOP-1:", top["text"][:n].replace("\n", " "))
    print("-" * 80)

for i in nc[nc["rank"] > 1]["i"].head(10):
    show(i, "Non-Compete", hybrid)

PACIRA PHARMACEUTICALS, INC. - A_R STRATEGIC LICENSING, DISTRIBUTION AND MARKETING AGREEMENT 
GOLD : During the Term, PPI and its Affiliates shall not: (i) file for Marketing Authorization with respect to any Competing Product in  any country in the Territory, (ii) manufacture or have manufactured any Competing Product in any country in the Territory, (iii) market or  have marketed any Competing Product in any country in the Territory or (iv) license any Third Party to do any of the foregoing.
TOP-1: o time;  "Commercial Launch"    Shall mean the date of the first arm's length sale by EKR to an unaffiliated Third Party customer for commercial  use of Product in a country within the Territory following the grant of Marketing Authorization and any  necessary pricing approval in that country;  "Commercialization  Committee"   Shall mean the committee to be set up under the terms of Article 5;         -4-  "Competing Product"    Means any [**] ([**] hours) [**] preparation (other than the 

In [4]:
def fully_contained(chunks, span):
    s, e = span
    return any(ch["start"] <= s and e <= ch["end"] for ch in chunks)

rows = []
for cl in TARGET_CLAUSES:
    total = split = 0
    for i in dev_idx:
        c = contracts[i]
        chunks = chunker(c["text"])
        for sp in c["gold"][cl]:
            total += 1
            split += not fully_contained(chunks, sp)
    rows.append({"clause": cl, "spans": total, "split_pct": round(split / total * 100, 1)})
pd.DataFrame(rows)

,clause,spans,split_pct
0,Governing Law,327,0.9
1,Renewal Term,140,4.3
2,Notice Period To Terminate Renewal,79,2.5
3,Termination For Convenience,164,1.8
4,Non-Compete,173,11.0
5,Cap On Liability,477,6.9
6,Uncapped Liability,130,9.2
